In [15]:
import sys
sys.path.append('..')
from src.bq_client import run_query
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [18]:
# Cohort Retention Matrix Query
with open('../src/queries/cohort_retention.sql', 'r') as f:
    cohort_sql = f.read()

df_cohort = run_query(cohort_sql)
print(f"Cohort data loaded! Shape: {df_cohort.shape}")


# MoM Revenue Growth Query
with open('../src/queries/monthly_revenue.sql', 'r') as f:
    revenue_sql = f.read()

print("Running monthly_revenue.sql against BigQuery...")
df_revenue = run_query(revenue_sql)
print(f"Revenue data loaded! Shape: {df_revenue.shape}")

Cohort data loaded! Shape: (91, 6)
Running monthly_revenue.sql against BigQuery...
Revenue data loaded! Shape: (90, 4)


In [19]:
print(" COHORT RETENTION MATRIX DATA FRAME ")
display(df_cohort.head())

 COHORT RETENTION MATRIX DATA FRAME 


,cohort_month,m0,m1,m2,m3,m6
0,2019-01-01,4,0,0,0,0
1,2019-02-01,12,1,0,0,0
2,2019-03-01,16,0,0,1,0
3,2019-04-01,29,1,0,0,0
4,2019-05-01,40,0,1,0,0


In [20]:

print("\n MONTH-OVER-MONTH REVENUE DATA FRAME")
display(df_revenue.head())


 MONTH-OVER-MONTH REVENUE DATA FRAME


,month,revenue,prev_month_revenue,MOM_growth_pct
0,2019-01-01,234.969997,NaN,NaN
1,2019-02-01,1574.809993,234.969997,570.22
2,2019-03-01,1778.570004,1574.809993,12.94
3,2019-04-01,1918.870004,1778.570004,7.89
4,2019-05-01,3695.930016,1918.870004,92.61


- Running Validation Tests on Cohort Matrix

In [21]:
assert df_cohort['cohort_month'].is_unique, "FAIL: Duplicate cohort months found!"
print("Check 1 Passed: Cohort months are completely unique.")

violating_cohorts = df_cohort[df_cohort['m1'] > df_cohort['m0']]
assert len(violating_cohorts) == 0, f"FAIL: Found anomalous cohorts where m1 > m0: {violating_cohorts['cohort_month'].tolist()}"
print("Check 2 Passed: Cohort decay constraints are statistically healthy.")

Check 1 Passed: Cohort months are completely unique.
Check 2 Passed: Cohort decay constraints are statistically healthy.


- Running Validation Tests on Month-Over-Month Revenue Data


In [22]:
assert df_revenue['month'].is_monotonic_increasing, "FAIL: Revenue timeline is not sorted chronologically!"
print("Check 1 Passed: Revenue timeline is strictly chronological.")

first_row_growth = df_revenue.iloc[0]['MOM_growth_pct']
assert pd.isna(first_row_growth) or first_row_growth is None, f"FAIL: First row growth rate should be Null, got {first_row_growth}"
print("Check 2 Passed: Boundary rows are safely handling division-by-zero boundaries.")


Check 1 Passed: Revenue timeline is strictly chronological.
Check 2 Passed: Boundary rows are safely handling division-by-zero boundaries.


In [23]:
df_retention_rates = df_cohort.copy()
df_retention_rates['cohort_month'] = df_retention_rates['cohort_month'].astype(str)

# target cols for retention rate calculations
milestone_cols = ['m1','m2','m3','m6']

# retention rate percentage relative to the initial baseline (m0)
for col in milestone_cols:
    df_retention_rates[f'{col}_rate'] = (df_retention_rates[col] / df_retention_rates['m0']) * 100

rate_columns = [f'{col}_rate' for col in milestone_cols]
df_retention_rates[rate_columns] = df_retention_rates[rate_columns].round(2)

structured_matrix = df_retention_rates[['cohort_month', 'm0'] + rate_columns]

print("RETENTION RATE MATRIX PROCESSING COMPLETE")
print("-" * 70)
display(structured_matrix.head(10))

RETENTION RATE MATRIX PROCESSING COMPLETE
----------------------------------------------------------------------


,cohort_month,m0,m1_rate,m2_rate,m3_rate,m6_rate
0,2019-01-01,4,0.0,0.0,0.0,0.0
1,2019-02-01,12,8.33,0.0,0.0,0.0
2,2019-03-01,16,0.0,0.0,6.25,0.0
3,2019-04-01,29,3.45,0.0,0.0,0.0
4,2019-05-01,40,0.0,2.5,0.0,0.0
5,2019-06-01,43,0.0,2.33,2.33,2.33
6,2019-07-01,52,0.0,1.92,0.0,5.77
7,2019-08-01,67,1.49,0.0,0.0,0.0
8,2019-09-01,72,0.0,1.39,0.0,4.17
9,2019-10-01,89,1.12,5.62,3.37,1.12


- Cohort Heatmap

In [24]:
rate_cols = ['m1_rate', 'm2_rate', 'm3_rate', 'm6_rate']
df_heatmap_input = df_retention_rates.copy()

# cleaning the date labels
df_heatmap_input['cohort_label'] = df_heatmap_input['cohort_month'].astype(str).str[:7]

fig_heatmap = px.imshow(
    df_heatmap_input[rate_cols].values,
    labels = dict(x = "Months After Acquisition", y = "Cohort", color = "Retention %"),
    x = ['Month 1', 'Month 2', 'Month 3', 'Month 6'],
    y = df_heatmap_input['cohort_label'].tolist(),
    color_continuous_scale = 'Blues',
    title = 'Cohort Retention Heatmap: % Returning Customers by Acquisition Month'
)

fig_heatmap.update_layout(height=600)
fig_heatmap.show()


- Revenue Chart

In [25]:
df_revenue_plot = df_revenue.copy()
df_revenue_plot['month'] = pd.to_datetime(df_revenue_plot['month'])

fig_revenue= go.Figure()

# Revenue Volume Bar Chart
fig_revenue.add_trace(
    go.Bar(
        x = df_revenue_plot['month'],
        y = df_revenue_plot['revenue'],
        name = "Monthly Revenue",
        marker_color = 'steelblue',
        opacity = 0.7
    )
)

# Growth Velocity Line Chart
fig_revenue.add_trace(
    go.Scatter(
        x = df_revenue_plot['month'],
        y = df_revenue_plot['MOM_growth_pct'],
        name = "MoM Growth %",
        yaxis = 'y2',
        line = dict(color = 'orange', width = 2),
        mode = 'lines+markers'
    )
)

# Dual-axis layout
fig_revenue.update_layout(
    title = 'Monthly Revenue & MoM Growth Rate',
    yaxis = dict(title='Revenue ($)'),
    yaxis2 = dict(
        title = 'MoM Growth %',
        overlaying = 'y',
        side = 'right',
        zeroline = True,
        zerolinecolor = 'red'
    ),
    legend = dict(x=0, y=1.1, orientation = 'h'),
    height = 450
)

fig_revenue.show()


In [26]:
avg_m1 = df_retention_rates['m1_rate'].mean()
avg_m3 = df_retention_rates['m3_rate'].mean()

latest_mom = df_revenue_plot['MOM_growth_pct'].dropna().iloc[-1]

print("=" * 70)
print("RETENTION DIAGNOSIS REPORT")
print("=" * 70)
print(f"Average Month-1 Retention:  {avg_m1:.2f}%")
print(f"Average Month-3 Retention:  {avg_m3:.2f}%")
print(f"Latest MoM Revenue Growth:  {latest_mom:.2f}%")
print("=" * 70)

RETENTION DIAGNOSIS REPORT
Average Month-1 Retention:  2.49%
Average Month-3 Retention:  2.00%
Latest MoM Revenue Growth:  31.29%
